# Colony counting pipeline

Isolate each well from a multi-well plate `.tif` scan, then run each well crop through Cellpose to count colonies and estimate diameter. Source scans live in `../Clonogenics`.

## 0. Colab setup — run once per session

Only needed the first time you open this notebook in Colab, or after a runtime
restart (Colab wipes the VM each time). Running locally with `uv`? Skip this cell.


In [ ]:
#@title Setup (Colab clone + install)
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os

    REPO_URL = "https://github.com/Districtfine/auto-clonogenics.git"
    REPO_DIR = "/content/auto-clonogenics"  # absolute -- re-running this cell from inside
    # the repo (cwd already moved) must not treat "auto-clonogenics" as a fresh relative
    # target and clone into itself again, which nests a new copy each re-run.

    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    # Colab already ships torch/opencv/numpy/pandas/scipy with GPU wiring intact --
    # only pull the packages this pipeline adds on top, so we don't clobber those.
    RESTART_MARKER = "/content/.setup_restarted"

    if not os.path.exists(RESTART_MARKER):
        # numba (pulled in transitively by cellpose/ultralytics) caps at numpy<2.1; letting
        # pip resolve to the newest numpy instead breaks that pin and leaves a mixed-version
        # set of numpy/scipy files on disk (a partial upgrade, not just a stale in-memory
        # import) -- that surfaces later as an ImportError deep in an unrelated cell. Pin
        # numpy up front and force scipy to rebuild against it, so the resolver can't drift.
        !pip install -q "numpy<2.1" cellpose ultralytics tifffile jupyter-bbox-widget gdown
        !pip install -q --force-reinstall "numpy<2.1" scipy
        open(RESTART_MARKER, "w").close()
        print("Installed. Restarting the runtime once so numpy/scipy reload cleanly -- "
              "just run this cell again once it restarts (nothing will reinstall this time).")
        os.kill(os.getpid(), 9)
else:
    print("Not running in Colab — skipping clone/install (using local uv environment).")


## How this notebook works

1. **Tune** (Sections 1–4): in the Config cell below, set `RUN_MODE = "Tune (single image)"`,
   then run every cell down through Section 4 on one scan. Check the printed preview and
   colony counts look right — re-run the Config cell with different values and re-run
   downstream cells as needed.
2. **Batch** (Section 5): once tuning looks good, set `RUN_MODE = "Batch (all scans)"` in
   the Config cell, re-run it, then run Section 5 — it sweeps every scan in `INPUT_DIR` and
   writes results + PNGs to `BATCH_OUTPUT_DIR`, zipped up for download.

You only draw ROI boxes once (Section 3c) — they're reused for every scan in the batch, as
long as the plate layout doesn't change.


In [ ]:
#@title Imports
import os
import glob
import shutil
import warnings
from datetime import datetime
warnings.filterwarnings("ignore", message="Sparse invariant checks")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tifi
import torch
from scipy import ndimage
from tqdm.notebook import tqdm
from ultralytics import FastSAM
from cellpose import models

plt.rcParams['figure.figsize'] = [12, 6]

## 1. Config

**1a (below): fill in every run** — dataset location and plate layout.
**1b (below that): tuning defaults** — usually fine as-is. Cellpose segmentation knobs are the ones worth adjusting per experiment; the well-detection constants further down aren't exposed as form fields at all (see that cell's own comment).

In [ ]:
#@title Dataset & plate layout (fill in every run)
#@markdown ### Run mode
#@markdown **Tune** runs `REFERENCE_SCAN` alone with plots shown, so you can check
#@markdown settings before committing to a batch. **Batch** sweeps every scan in
#@markdown `INPUT_DIR` (Section 3c) and writes results/PNGs to disk.
RUN_MODE = "Tune (single image)"  #@param ["Tune (single image)", "Batch (all scans)"]
BATCH_MODE = RUN_MODE.startswith("Batch")

#@markdown ---
#@markdown ### Dataset location
#@markdown Colab only (ignored when running locally with uv). For a shared folder link,
#@markdown the folder must be shared "Anyone with the link".
DATA_SOURCE = "Shared folder link"  #@param ["Shared folder link", "Upload files"]
FOLDER_URL = ""  #@param {type:"string"}

#@markdown ---
#@markdown ### Plate layout
PLATE_ROWS = None  #@param {type:"raw"}
PLATE_COLS = None  #@param {type:"raw"}


In [ ]:
#@title Tuning defaults (usually fine as-is)
BATCH_OUTPUT_DIR = "batch_output"  #@param {type:"string"}
# Placement-shift tolerance: how far a plate may sit from where it was drawn, as a
# fraction of the drawn box. This is a property of hand-placing trays in a scanner, not of
# any one dataset, so it is set generously; resolve_axis is what stops a wide search window
# from accepting a wrong edge.
MARGIN_FRAC = 0.20  #@param {type:"slider", min:0.0, max:0.5, step:0.01}

# ============================================================
# Well-detection constants (scan contrast, Hough ring-snap, rigid-geometry tolerances).
# Not exposed as form fields — these tune plate/well *finding*, not colony segmentation,
# and shouldn't need touching unless detection itself misbehaves on a new scan style.
# Edit the values directly (Colab: View > Editor view, or expand this cell) if it does.
# ============================================================
WELLS_ONLY = False  # True: stop after well-detection, skip the slower Cellpose step.
CLAHE_CLIP = 3.0

# Per-well edge refinement (local Hough): the analytic layout is close but not
# pixel-perfect; snap each circle to the real ring via Hough in a small ROI around the
# computed center, constrained near the known radius. Falls back to the analytic circle
# if no good ring is found. Downscaling the ROI before searching (then scaling the found
# circle back up) gives ~12x speedup for a few px of localization noise.
REFINE_WELLS = True
REFINE_SEARCH_FRAC = 0.5
REFINE_RADIUS_TOL = 0.2
REFINE_MAX_SHIFT = 0.5
REFINE_DOWNSCALE_PX = 300
HOUGH_PARAM1 = 100
HOUGH_PARAM2 = 30
# Seed radius for that ring search, as a fraction of the grid pitch (the spacing between
# neighbouring wells). Wells nearly fill their grid cell whatever the format, so one number
# covers a moulded 3x2 tray and a column of loose petri dishes — no per-layout radius. It only
# seeds the search: the radius actually used comes from the rings that lock.
WELL_RADIUS_PITCH_FRAC = 0.45

# Rigid-geometry tolerances. A plate can move between scans but cannot change size, so a
# detected box may differ from the drawn one in position but not in dimensions.
# PLATE_SIZE_TOL: how much size disagreement between two snapped plate edges is still
# blamed on edge-localisation noise rather than on one edge having locked onto the wrong
# thing (a neighbouring tray, an interior rib).
# GRID_PITCH_TOL is the same idea one level down — how far the well spacing fitted from the
# wells that locked may stray from the spacing implied by the plate box before the fit is
# judged bogus and only the grid's origin is taken from the locks.
# MAX_GRID_ROTATION_DEGREES: how skew a tray may lie in the scanner before a fitted well
# grid is judged bogus — trays do sit a degree or two off square, they do not sit sideways.
# LOCK_TRUST_FRAC: how far a well's own ring lock may sit from the fitted grid and still be
# believed, as a fraction of the well pitch. Beyond it the lock is something else in the well
# — a dense colony mass reads as a circle — and the grid position is used instead.
PLATE_SIZE_TOL = 0.15
GRID_PITCH_TOL = 0.25
MAX_GRID_ROTATION_DEGREES = 8.0
LOCK_TRUST_FRAC = 0.25

# ============================================================
# Cellpose colony segmentation. Tune these if colonies look under/over-segmented (two
# touching colonies merged into one blob, or one colony split into two). Full parameter
# docs: https://cellpose.readthedocs.io/en/latest/settings.html
# (API reference for the eval() call these feed: https://cellpose.readthedocs.io/en/latest/api.html)
# ============================================================
#@markdown ---
#@markdown ### Cellpose colony segmentation
#@markdown Parameter docs: [cellpose.readthedocs.io/settings](https://cellpose.readthedocs.io/en/latest/settings.html)
# Expected colony diameter in px, in the LAB-distance "signal" image fed to Cellpose (not
# the raw well crop). Cellpose uses this to scale its internal model — too large merges
# nearby colonies, too small can shatter one colony into several. A well crop on these scans
# is ~1900 px across and a distinct colony ~80-120 px, so this is about one colony; raising
# it is the knob that makes a cloud of specks read as a single object rather than a crowd.
#@markdown **Colony diameter (px):** too large merges touching colonies; too small splits one colony into several.
CELLPOSE_DIAMETER = 70  #@param {type:"integer"}
# Max allowed flow-reconstruction error per mask (Cellpose's internal QC score). Higher =
# keep more masks, including rougher/noisier ones; lower = reject malformed masks more
# aggressively (fewer false positives, but can also drop real irregular colonies).
#@markdown **Flow threshold:** higher keeps more (rougher) masks; lower rejects malformed ones more aggressively.
CELLPOSE_FLOW_THRESHOLD = 0.9  #@param {type:"slider", min:0.0, max:3.0, step:0.1}
# A pixel is called "part of a colony" where the model's cell-probability map exceeds this.
# Lower (more negative) = more permissive, catches faint/sparse colonies but risks noise;
# higher = stricter, cleaner background but can miss faint real colonies.
#@markdown **Cell-probability threshold:** lower (more negative) catches faint colonies but risks noise; higher is stricter.
CELLPOSE_CELLPROB_THRESHOLD = -2.0  #@param {type:"slider", min:-6.0, max:6.0, step:0.5}
# Percentile range used to normalize the signal image's intensity before segmentation —
# clips extreme outlier pixels so one bright artifact doesn't wash out the contrast Cellpose
# needs to see real colonies.
#@markdown **Normalize percentile range:** clips extreme-outlier pixels before segmentation.
CELLPOSE_NORM_LOW = 1.0  #@param {type:"slider", min:0.0, max:10.0, step:0.5}
CELLPOSE_NORM_HIGH = 99.0  #@param {type:"slider", min:90.0, max:100.0, step:0.5}
CELLPOSE_NORMALIZE_PERCENTILE = [CELLPOSE_NORM_LOW, CELLPOSE_NORM_HIGH]
# Smallest thing still counted as a colony, given as its diameter in px of the well crop and
# converted to the mask *area* Cellpose wants (min_size). This is the pin-prick filter: a
# single-cell fleck reads ~10 px across, far under any real colony, and no probability
# threshold rejects it as reliably as an outright size cut — a speck can be perfectly
# confident and still not be a colony. Because Cellpose runs its dynamics at the original crop
# resolution (resample=True), this is in raw crop pixels, not rescaled ones. Raise it if specks
# still get counted; lower it if genuinely small but real colonies start disappearing.
#@markdown **Minimum colony diameter (px):** filters out single-cell specks; raise if debris still counts, lower if small real colonies vanish.
CELLPOSE_MIN_COLONY_DIAMETER = 15  #@param {type:"integer"}
CELLPOSE_MIN_SIZE = int(np.pi * (CELLPOSE_MIN_COLONY_DIAMETER / 2) ** 2)
# Non-colony brightness outlier rejection, in LAB L (lightness) units, relative to the local
# well-background L — not tied to position, since glints/debris can land anywhere in a well.
# Crystal-violet colonies sit in a mid-darkness band: dirt/hair/specks are much darker
# (near-black) than any real colony, while a specular reflection/glint is brighter than
# background. (dark_margin, bright_margin): a pixel darker than background by more than
# dark_margin, or brighter than background by more than bright_margin, is zeroed out of the
# Cellpose signal image before segmentation. Raise dark_margin if real dense/dark colonies
# start getting stripped; raise bright_margin if real bright-background wells start losing
# edge pixels; lower either if debris/glint still leaks through as false colonies.
#@markdown ---
#@markdown ### Debris/glint filtering (pre-Cellpose, not a Cellpose parameter)
#@markdown Rejects outlier pixels from the signal image *before* Cellpose ever sees it:
#@markdown dark = debris cutoff, bright = glint cutoff, relative to well background.
#@markdown Higher = more permissive (only the most extreme pixels on that side get
#@markdown rejected); lower = stricter (rejects more, risking real colony pixels too).
L_DARK_MARGIN = 97  #@param {type:"slider", min:0, max:150, step:1}
L_BRIGHT_MARGIN = 40  #@param {type:"slider", min:0, max:150, step:1}
L_OUTLIER_MARGIN = (L_DARK_MARGIN, L_BRIGHT_MARGIN)  # (dark_margin, bright_margin)

print("Config loaded. Plate geometry now comes from ROI hints (Section 3c), not a profile.")


## 2. Pick device & load models

Picks `mps`/`cuda`/`cpu` automatically (portable between Apple Silicon and the 1650 Ti box), loads FastSAM and Cellpose (colony counter) once. Reused by both single-image and batch runs below.

In [ ]:
#@title Pick device & load models
# Portable device pick: mps on Apple Silicon, cuda on the 1650 Ti box, cpu fallback.
# DEVICE (str) feeds ultralytics/FastSAM; TORCH_DEVICE (torch.device) feeds Cellpose.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
TORCH_DEVICE = torch.device(DEVICE)
print(f"Using device: {DEVICE}")

if "cp_model" in globals():
    # Reloading these is slow (weights + GPU transfer) and pointless once loaded --
    # "Run this cell and below" after tweaking a Cellpose param shouldn't reload the
    # models too. Restart the runtime if you actually need a clean reload.
    print("Models already loaded, reusing.")
else:
    print("Loading FastSAM (Well Extractor) into GPU...")
    sam_model = FastSAM('FastSAM-s.pt')

    print("Loading Cellpose (Colony Counter) into GPU...")
    cp_model = models.CellposeModel(gpu=DEVICE != "cpu", device=TORCH_DEVICE, model_type='cpsam_v2')
    # Directly confirm what device the model actually landed on — don't infer this from
    # whether Cellpose's own log messages showed up, since those need io.logger_setup()
    # (which also drives a per-tile progress bar) to be visible at all.
    print(f"cp_model.device = {cp_model.device}, cp_model.gpu = {cp_model.gpu}")
    print("Done!")


## 3. Detect wells (ROI-hinted)

Well detection is driven by the ROI hints drawn in Section 3c — no per-dataset profile, no global crop. `refine_well` below (the per-well Hough ring-snap) is the one piece kept from the old approach; it is reused by the ROI pipeline in Section 3b/3c. Run this cell to define it.

In [ ]:
#@title Well-ring refinement
def refine_well(gray, cx, cy, r):
    """Snap an analytic (cx, cy, r) to the real well ring with a local Hough search.

    gray is the full-image grayscale. Runs Hough on a downscaled ROI (much faster on these
    high-res scans) and scales the result back up. Returns (cx, cy, r, locked); when locked
    is False no ring near the prior was found and the input is returned unchanged, so the
    caller can place that well from its neighbours instead of trusting a stale position
    (see place_wells). Reused by the ROI-hinted path below.
    """
    pad = int(r * (1 + REFINE_SEARCH_FRAC))
    height, width = gray.shape
    x0, x1 = max(0, cx - pad), min(width, cx + pad)
    y0, y1 = max(0, cy - pad), min(height, cy + pad)
    roi = gray[y0:y1, x0:x1]
    if roi.size == 0:
        return cx, cy, r, False

    roi = cv2.medianBlur(roi, 5)
    scale = min(1.0, REFINE_DOWNSCALE_PX / max(roi.shape))
    search_roi = cv2.resize(roi, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1.0 else roi

    circles = cv2.HoughCircles(
        search_roi, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=max(search_roi.shape),          # only expect one ring in this ROI
        param1=HOUGH_PARAM1, param2=HOUGH_PARAM2,
        minRadius=int(r * (1 - REFINE_RADIUS_TOL) * scale),
        maxRadius=int(r * (1 + REFINE_RADIUS_TOL) * scale),
    )
    if circles is None:
        return cx, cy, r, False
    if scale < 1.0:
        circles = circles / scale  # back to ROI (full-res) coordinates

    # pick the ring whose center is closest to the analytic prior (ROI center)
    roi_cx, roi_cy = cx - x0, cy - y0
    found_cx, found_cy, found_r = min(
        circles[0], key=lambda cir: (cir[0] - roi_cx) ** 2 + (cir[1] - roi_cy) ** 2
    )
    if (found_cx - roi_cx) ** 2 + (found_cy - roi_cy) ** 2 > (r * REFINE_MAX_SHIFT) ** 2:
        return cx, cy, r, False  # moved too far — likely a colony/edge, not this well's ring

    return int(x0 + found_cx), int(y0 + found_cy), int(found_r), True

## 3b. ROI-hinted plate detection

The user draws one box per plate **on the true plate outline** (`jupyter-bbox-widget`, Section 3c); code pads it by `MARGIN_FRAC` and re-detects the precise plate box per image, so hand-placement shift between scans is absorbed. Wells are placed by an even `rows×cols` grid inside the detected box, then corrected by the wells that lock onto a real ring (`place_wells` + `refine_well`, Section 3).

`detect_plate_rect_edges` sums gradient strength *along* each edge direction and snaps each of the four sides to the strong line nearest the drawn one. The two edges of an axis are then reconciled as a **rigid plate** (`resolve_axis`): a plate can move between scans but cannot change size, so two edges implying a different size than the drawn box means one of them locked onto something else — a neighbouring tray, an interior rib — and the axis is translated by the more confident edge instead. An axis where neither edge was found keeps the drawn edge and says so, since that box still describes the *reference* scan.

(An earlier CLAHE→Canny→contour detector was raced against this one; it consistently grabbed the tray flange and drifted the wells, so it was dropped.)

In [ ]:
#@title Grid & box helpers
def well_grid_fracs(rows, cols):
    """Even-spaced cell-center fractions inside a plate box (center of each grid cell)."""
    x_fracs = [(col_index + 0.5) / cols for col_index in range(cols)]
    y_fracs = [(row_index + 0.5) / rows for row_index in range(rows)]
    return x_fracs, y_fracs


def roi_frac_to_px(roi, image_width, image_height):
    """Convert a fractional {x,y,w,h} box (each in [0,1]) to integer (x0,y0,x1,y1) pixels."""
    x0 = int(round(roi["x"] * image_width))
    y0 = int(round(roi["y"] * image_height))
    x1 = int(round((roi["x"] + roi["w"]) * image_width))
    y1 = int(round((roi["y"] + roi["h"]) * image_height))
    return x0, y0, x1, y1


def pad_box(box_px, image_width, image_height, margin_frac):
    """Expand a pixel box by margin_frac of its own width/height on each side, clamped."""
    x0, y0, x1, y1 = box_px
    pad_x = int(round((x1 - x0) * margin_frac))
    pad_y = int(round((y1 - y0) * margin_frac))
    return (
        max(0, x0 - pad_x),
        max(0, y0 - pad_y),
        min(image_width, x1 + pad_x),
        min(image_height, y1 + pad_y),
    )

In [ ]:
#@title Plate-edge detection
def snap_edge(strength_profile, prior_index, radius, min_peak_ratio):
    """Snap to the strong edge NEAREST the drawn (prior) edge within +/- radius.

    Since the user draws accurately on the true plate, the plate's own wall is the closest
    strong edge to the drawn line; a neighbouring plate's wall (or the handwriting plate to
    the side) is farther, so 'nearest strong edge' rejects it even when it is brighter.
    A column/row counts as an edge if its summed gradient exceeds min_peak_ratio x the
    profile median. Returns (index, confidence): confidence is the chosen line's ratio to
    the profile median, and 0.0 means nothing in the window qualified. The caller picks the
    fallback — silently keeping the drawn edge anchors the box to where the plate *was* on
    the reference scan, which must not pass unnoticed.
    """
    profile_median = float(np.median(strength_profile)) or 1.0
    threshold = min_peak_ratio * profile_median
    low = max(0, prior_index - radius)
    high = min(len(strength_profile), prior_index + radius + 1)
    strong_indices = [index for index in range(low, high) if strength_profile[index] > threshold]
    if not strong_indices:
        return prior_index, 0.0
    nearest = min(strong_indices, key=lambda index: abs(index - prior_index))
    return nearest, float(strength_profile[nearest]) / profile_median


def resolve_axis(low_snap, high_snap, prior_low, prior_high, size_tol):
    """Combine the two snapped edges of one axis, treating the plate as rigid.

    A plate cannot change size between scans, it can only move. So both snapped edges are
    trusted only when the size they imply matches the drawn size within size_tol; otherwise
    at least one of them locked onto something that is not this plate's wall, and the axis is
    instead *translated* by whichever edge was more confident, keeping the drawn size. If
    neither edge was found the drawn axis is kept and reported as "none" — that box is
    anchored to the reference scan, not this one. Returns (low, high, status).
    """
    low_index, low_confidence = low_snap
    high_index, high_confidence = high_snap
    prior_size = prior_high - prior_low

    if low_confidence and high_confidence:
        if abs((high_index - low_index) - prior_size) <= size_tol * prior_size:
            return low_index, high_index, "both"
    if low_confidence and low_confidence >= high_confidence:
        return low_index, low_index + prior_size, "low"
    if high_confidence:
        return high_index - prior_size, high_index, "high"
    return prior_low, prior_high, "none"


def detect_plate_rect_edges(gray, search_box, prior_box, min_peak_ratio=2.0,
                            size_tol=PLATE_SIZE_TOL):
    """Snap each of the four plate edges to the strong line NEAREST the drawn edge.

    Sums |gradient| along each edge direction (down columns for the vertical left/right edges,
    across rows for the horizontal top/bottom edges). Each edge is searched only within +/- the
    padding on that side (the placement-shift tolerance) and snapped to the nearest strong
    line, so it can't jump to a neighbouring plate or the handwriting plate. The two edges of
    an axis are then reconciled as a rigid plate (resolve_axis), which is what keeps a
    generous search window from silently accepting an impossible box. Returns
    ((x, y, w, h) in full-image pixels, {"x": status, "y": status}).
    """
    search_x0, search_y0, search_x1, search_y1 = search_box
    prior_x0, prior_y0, prior_x1, prior_y1 = prior_box
    crop = gray[search_y0:search_y1, search_x0:search_x1]

    gradient_x = np.abs(cv2.Sobel(crop, cv2.CV_64F, 1, 0, ksize=3))
    gradient_y = np.abs(cv2.Sobel(crop, cv2.CV_64F, 0, 1, ksize=3))
    column_strength = gradient_x.sum(axis=0)   # one value per column -> vertical edges
    row_strength = gradient_y.sum(axis=1)      # one value per row    -> horizontal edges

    # search radius on each side = the padding on that side (i.e. the shift tolerance)
    radius_left = max(1, prior_x0 - search_x0)
    radius_right = max(1, search_x1 - prior_x1)
    radius_top = max(1, prior_y0 - search_y0)
    radius_bottom = max(1, search_y1 - prior_y1)

    left_snap = snap_edge(column_strength, prior_x0 - search_x0, radius_left, min_peak_ratio)
    right_snap = snap_edge(column_strength, prior_x1 - search_x0, radius_right, min_peak_ratio)
    top_snap = snap_edge(row_strength, prior_y0 - search_y0, radius_top, min_peak_ratio)
    bottom_snap = snap_edge(row_strength, prior_y1 - search_y0, radius_bottom, min_peak_ratio)

    left_local, right_local, x_status = resolve_axis(
        left_snap, right_snap, prior_x0 - search_x0, prior_x1 - search_x0, size_tol)
    top_local, bottom_local, y_status = resolve_axis(
        top_snap, bottom_snap, prior_y0 - search_y0, prior_y1 - search_y0, size_tol)

    left_x = search_x0 + left_local
    right_x = search_x0 + right_local
    top_y = search_y0 + top_local
    bottom_y = search_y0 + bottom_local
    return (left_x, top_y, right_x - left_x, bottom_y - top_y), {"x": x_status, "y": y_status}

In [ ]:
#@title Well placement
def reconcile_vertical_borders(gray, plate_boxes, prior_boxes, margin_frac=0.05, min_peak_ratio=2.0):
    """Make vertically-stacked plates share one hard border, so they never gap or overlap.

    The two plates sit in one molded tray with a single rib between them. Detecting each
    plate's inner edge independently lets them disagree on a shifted scan (gap or overlap).
    Instead, for each adjacent pair we find the single strongest horizontal edge (the rib)
    in the band between their drawn inner edges (widened by the shift tolerance) and set the
    upper plate's bottom = the lower plate's top = that line. Mirrors the old detector's
    awareness of the hard border between stacked plates. Returns updated (x, y, w, h) boxes.
    """
    order = sorted(range(len(plate_boxes)), key=lambda index: plate_boxes[index][1])
    boxes = [list(box) for box in plate_boxes]
    for upper_index, lower_index in zip(order, order[1:]):
        # shared horizontal extent of the two boxes (only look at columns they both cover)
        shared_x0 = max(boxes[upper_index][0], boxes[lower_index][0])
        shared_x1 = min(boxes[upper_index][0] + boxes[upper_index][2],
                        boxes[lower_index][0] + boxes[lower_index][2])
        if shared_x1 <= shared_x0:
            continue

        drawn_upper_bottom = prior_boxes[upper_index][3]
        drawn_lower_top = prior_boxes[lower_index][1]
        upper_drawn_height = prior_boxes[upper_index][3] - prior_boxes[upper_index][1]
        lower_drawn_height = prior_boxes[lower_index][3] - prior_boxes[lower_index][1]
        radius = int(max(upper_drawn_height, lower_drawn_height) * margin_frac)
        band_lo = max(0, min(drawn_upper_bottom, drawn_lower_top) - radius)
        band_hi = min(gray.shape[0], max(drawn_upper_bottom, drawn_lower_top) + radius)
        default_border_y = (drawn_upper_bottom + drawn_lower_top) // 2

        band = gray[band_lo:band_hi, shared_x0:shared_x1]
        if band.size == 0:
            border_y = default_border_y
        else:
            gradient_y = np.abs(cv2.Sobel(band, cv2.CV_64F, 0, 1, ksize=3))
            row_strength = gradient_y.sum(axis=1)
            profile_median = float(np.median(row_strength)) or 1.0
            peak = int(np.argmax(row_strength))
            border_y = band_lo + peak if row_strength[peak] > min_peak_ratio * profile_median else default_border_y

        # snap the shared edge: upper plate ends at border_y, lower plate starts at border_y
        boxes[upper_index][3] = max(1, border_y - boxes[upper_index][1])
        lower_bottom = boxes[lower_index][1] + boxes[lower_index][3]
        boxes[lower_index][1] = border_y
        boxes[lower_index][3] = max(1, lower_bottom - border_y)
    return [tuple(box) for box in boxes]


def fit_grid_axis(locked_indices, locked_centers, analytic_pitch, analytic_origin,
                  pitch_tol=GRID_PITCH_TOL):
    """Fit one axis of the well grid — origin and pitch — from the wells that locked.

    Wells are moulded on a regular grid, so along either axis a centre is
    origin + index * pitch: a two-parameter line fit over the grid indices. Fitting it from
    the locked wells is what stops the *drawn* box from setting the well spacing — a box drawn
    loose or tight then only changes the starting guess, not the answer.

    Two distinct indices are needed to fit a pitch at all; with fewer, the analytic pitch is
    kept and only the origin is re-centred on the locks. A fitted pitch straying further than
    pitch_tol from the analytic one is rejected the same way, so a couple of bad locks cannot
    invent a grid of their own. Returns (origin, pitch).
    """
    indices = np.asarray(locked_indices, dtype=float)
    centers = np.asarray(locked_centers, dtype=float)
    if len(indices) == 0:
        return analytic_origin, analytic_pitch

    recentered_origin = float(centers.mean() - indices.mean() * analytic_pitch)
    if len(np.unique(indices)) < 2:
        return recentered_origin, analytic_pitch

    pitch, origin = np.polyfit(indices, centers, 1)
    if not (1 - pitch_tol) <= pitch / analytic_pitch <= (1 + pitch_tol):
        return recentered_origin, analytic_pitch
    return float(origin), float(pitch)


def fit_grid_transform(locked_cols, locked_rows, locked_points,
                       analytic_pitch_x, analytic_pitch_y):
    """Least-squares affine mapping grid index (col, row) -> pixel centre; None if not usable.

    fit_grid_axis solves x against the column and y against the row independently, which
    cannot represent a tray lying slightly skew in the scanner: a rotation makes x depend on
    the *row* as well, so a per-axis fit stays internally consistent while drifting at the far
    corners. Fitting one 2x3 transform over the locked wells
    absorbs the rotation instead of leaving it in the residuals.

    Returns None — and the caller falls back to the per-axis fit — when the locks do not span
    two rows and two columns (the fit is underdetermined), when either implied pitch strays
    past GRID_PITCH_TOL of the pitch the plate box implies, or when the implied rotation
    exceeds MAX_GRID_ROTATION_DEGREES.
    """
    cols = np.asarray(locked_cols, dtype=float)
    rows = np.asarray(locked_rows, dtype=float)
    points = np.asarray(locked_points, dtype=float)
    if len(points) < 3 or len(np.unique(cols)) < 2 or len(np.unique(rows)) < 2:
        return None

    design = np.column_stack([cols, rows, np.ones(len(cols))])
    solution, *_ = np.linalg.lstsq(design, points, rcond=None)
    transform = solution.T                      # rows: [a, b, tx], [c, d, ty]

    # the two columns of the 2x2 part are the step per column and the step per row, so their
    # lengths are the fitted pitches and their angles are how far the grid sits off square
    column_step, row_step = transform[:, 0], transform[:, 1]
    pitch_x = float(np.hypot(*column_step))
    pitch_y = float(np.hypot(*row_step))
    if pitch_x <= 0 or pitch_y <= 0:
        return None
    if not (1 - GRID_PITCH_TOL) <= pitch_x / analytic_pitch_x <= (1 + GRID_PITCH_TOL):
        return None
    if not (1 - GRID_PITCH_TOL) <= pitch_y / analytic_pitch_y <= (1 + GRID_PITCH_TOL):
        return None

    column_rotation = np.degrees(np.arctan2(column_step[1], column_step[0]))
    row_rotation = np.degrees(np.arctan2(row_step[0], row_step[1]))
    if (abs(column_rotation) > MAX_GRID_ROTATION_DEGREES
            or abs(row_rotation) > MAX_GRID_ROTATION_DEGREES):
        return None
    return transform


def place_wells(refine_gray, plate_box, rows, cols, plate_letter, refine=True):
    """Wells on a fitted rows x cols grid, anchored by whichever wells lock onto a real ring.

    The analytic grid from the plate box is only a starting guess: it assumes wells sit at the
    centre of even grid cells, which no moulded tray actually does (there is a wider flange on
    one side), and it inherits any slop in how the ROI was drawn. So each well is Hough-snapped
    to its ring, and the wells that lock are fitted as one grid — by preference a single affine
    transform, which also absorbs a skew tray, and per-axis when there are too few locks to
    pin that down.

    The grid is not the last word, though: a well that found its own rim keeps that position,
    because the grid only describes wells that are actually moulded at a fixed pitch. Loose
    petri dishes laid out by hand in a roughly rectangular pattern are not, and forcing them
    collinear and evenly spaced shifts every dish. A lock further than LOCK_TRUST_FRAC of the
    pitch from the grid is not believed — that is a dense colony mass reading as a circle, not
    a rim — and wells that never locked are placed from the grid, so an empty well is still
    positioned by its neighbours rather than by a stale guess.

    Radius comes from the median locked ring, falling back to the analytic radius when nothing
    locked, and is capped so two wells can never overlap.
    """
    plate_x, plate_y, plate_width, plate_height = plate_box
    x_fracs, y_fracs = well_grid_fracs(rows, cols)
    # Seed the ring search from the grid pitch rather than the plate width, so the same number
    # works for a 3x2 moulded tray and a 3x1 column of loose dishes (see WELL_RADIUS_PITCH_FRAC).
    base_radius = int(WELL_RADIUS_PITCH_FRAC * min(plate_width / cols, plate_height / rows))

    analytic_centers, labels, row_indices, col_indices = [], [], [], []
    for row_index, y_frac in enumerate(y_fracs):
        for col_index, x_frac in enumerate(x_fracs):
            analytic_centers.append((plate_x + plate_width * x_frac,
                                     plate_y + plate_height * y_frac))
            labels.append(f"{plate_letter}{row_index * cols + col_index + 1}")
            row_indices.append(row_index)
            col_indices.append(col_index)

    if not refine:
        return [(int(center_x), int(center_y), base_radius, label)
                for (center_x, center_y), label in zip(analytic_centers, labels)]

    found_by_index = {}
    locked_points, locked_radii, locked_rows, locked_cols = [], [], [], []
    for index, (center_x, center_y) in enumerate(analytic_centers):
        found_x, found_y, found_r, locked = refine_well(
            refine_gray, int(center_x), int(center_y), base_radius)
        if locked:
            found_by_index[index] = (found_x, found_y)
            locked_points.append((found_x, found_y))
            locked_radii.append(found_r)
            locked_rows.append(row_indices[index])
            locked_cols.append(col_indices[index])

    analytic_pitch_x = plate_width * (x_fracs[1] - x_fracs[0]) if cols > 1 else plate_width
    analytic_pitch_y = plate_height * (y_fracs[1] - y_fracs[0]) if rows > 1 else plate_height

    transform = fit_grid_transform(locked_cols, locked_rows, locked_points,
                                   analytic_pitch_x, analytic_pitch_y) if locked_points else None
    if transform is None:
        origin_x, pitch_x = fit_grid_axis(locked_cols, [point[0] for point in locked_points],
                                          analytic_pitch_x, analytic_centers[0][0])
        origin_y, pitch_y = fit_grid_axis(locked_rows, [point[1] for point in locked_points],
                                          analytic_pitch_y, analytic_centers[0][1])
        centers = np.array([[origin_x + col_index * pitch_x, origin_y + row_index * pitch_y]
                            for row_index, col_index in zip(row_indices, col_indices)])
    else:
        grid_indices = np.column_stack([col_indices, row_indices, np.ones(len(row_indices))])
        centers = grid_indices @ transform.T

    # Prefer each well's own rim over the grid's prediction for it: the grid exists to place
    # the wells that found nothing, not to overrule the ones that did.
    trust_radius = LOCK_TRUST_FRAC * min(analytic_pitch_x, analytic_pitch_y)
    for index, found_point in found_by_index.items():
        disagreement = float(np.hypot(found_point[0] - centers[index][0],
                                      found_point[1] - centers[index][1]))
        if disagreement <= trust_radius:
            centers[index] = found_point

    radius = int(np.median(locked_radii)) if locked_radii else base_radius

    # Two wells cannot overlap, whatever ring Hough locked onto, so the closest pair of placed
    # centers is a hard ceiling on the radius.
    # Without it an oversized ring pulls a neighbouring well's colonies into this well's crop
    # and double-counts them.
    if len(centers) > 1:
        offsets = centers[:, None, :] - centers[None, :, :]
        distances = np.hypot(offsets[..., 0], offsets[..., 1])
        np.fill_diagonal(distances, np.inf)
        radius = min(radius, int(distances.min() // 2))

    return [(int(center_x), int(center_y), radius, label)
            for (center_x, center_y), label in zip(centers, labels)]


def detect_wells_from_rois(image_rgb, roi_hints, margin_frac,
                           refine=True, return_boxes=False):
    """ROI-hinted replacement for detect_plate_rects + x_limit_frac: detect a plate box per
    drawn ROI, reconcile the shared border between stacked plates, place wells, snap to rings.
    Returns (x, y, r, label) for every well.

    return_boxes=True also returns the detected (x, y, w, h) plate box per ROI, so overlays
    can show what the detector actually locked onto (vs the padded search area).
    """
    image_height, image_width = image_rgb.shape[:2]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY) if image_rgb.ndim == 3 else image_rgb
    refine_gray = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8)).apply(gray) if refine else gray

    # Pass 1: detect each plate box independently within its padded ROI.
    prior_boxes = [roi_frac_to_px(roi, image_width, image_height) for roi in roi_hints]
    plate_boxes = []
    for roi, prior_box in zip(roi_hints, prior_boxes):
        search_box = pad_box(prior_box, image_width, image_height, margin_frac)
        plate_box, status = detect_plate_rect_edges(gray, search_box, prior_box)
        plate_boxes.append(plate_box)
        # A box that found no edge is the drawn box, i.e. it still describes the *reference*
        # scan. Say so — that is the failure that otherwise looks like a merely sloppy grid.
        stale_axes = sorted(axis for axis, axis_status in status.items() if axis_status == "none")
        if stale_axes:
            print(f"  !! plate {roi.get('letter', '?')}: no plate edge found on "
                  f"{'/'.join(stale_axes)} — box kept from the drawn ROI, so it is anchored "
                  f"to the reference scan, not this one. Raise MARGIN_FRAC if the plate can "
                  f"move further than the current search window.")

    # Pass 2: make vertically-stacked plates share the hard border (no gap/overlap).
    if len(plate_boxes) > 1:
        plate_boxes = reconcile_vertical_borders(gray, plate_boxes, prior_boxes, margin_frac)

    # Pass 3: place + refine wells inside the reconciled boxes.
    all_wells = []
    for roi_index, (roi, plate_box) in enumerate(zip(roi_hints, plate_boxes)):
        plate_letter = roi.get("letter") or chr(ord("A") + roi_index)
        all_wells.extend(
            place_wells(refine_gray, plate_box, roi["rows"], roi["cols"],
                        plate_letter, refine=refine)
        )
    if return_boxes:
        return all_wells, plate_boxes
    return all_wells

## 3c. Draw the ROI hints

**How to run:**
1. Run the dataset cell, then the cell right after it — for a shared folder link it downloads the folder and shows a Scan dropdown to pick from; for upload it prompts you to pick files directly.
2. Run the bbox cell — **draw one box per plate** on the true plate outline (top plate first, then bottom).
3. Run the `roi_hints` cell to convert those boxes to image fractions.
4. Run Section 5 to sweep the scans; each renders a grid overlay (yellow search area, lime detected plate, red wells) and prints a `!!` line for any plate whose box could not be found in this scan.

The boxes are drawn once, on one scan, and reused for every later scan of the same plate layout — that is the whole point of the ROI hint. Redraw only when the layout itself changes.

In [ ]:
#@title Dataset
# ============================================================
# Dataset — which scans to run. ROI hints are drawn once on REFERENCE_SCAN (below) and
# reused for every scan in INPUT_DIR, so one folder must hold one plate layout; change
# DATA_SOURCE/FOLDER_URL in the Config cell (Section 1) and redraw to switch datasets.
# ============================================================
if not IN_COLAB:
    INPUT_DIR = "../Clonogenics"           # <-- edit to your local scans folder
    REFERENCE_SCAN = "reference_scan.tif"  # <-- edit to your reference scan filename
    BATCH_SCANS = []

elif DATA_SOURCE == "Shared folder link":
    # Downloads a Drive folder shared as "Anyone with the link" straight to the VM --
    # no OAuth, no whole-account grant, only the one folder pointed to by FOLDER_URL.
    if not FOLDER_URL:
        raise ValueError(
            "FOLDER_URL is empty -- paste your shared folder's link into the Config "
            "cell (Section 1) and re-run it before running this cell."
        )

    import gdown
    import ipywidgets as widgets
    from IPython.display import display

    shared_download_dir = "/content/shared_scans"
    os.makedirs(shared_download_dir, exist_ok=True)

    if globals().get("_downloaded_folder_url") == FOLDER_URL:
        # Same folder as last time -- reuse what's on disk instead of re-downloading
        # everything just because a Cellpose param changed and this cell re-ran.
        downloaded = sorted(
            name for name in os.listdir(shared_download_dir)
            if name.lower().endswith((".tif", ".tiff"))
        )
        print(f"Already downloaded from this folder ({len(downloaded)} scans) -- reusing.")
    else:
        # List the folder first rather than downloading everything: only .tif/.tiff are
        # scans (folders synced from a Mac often carry .DS_Store etc alongside them, which
        # aren't part of the dataset), and downloading files one at a time lets one file
        # that trips Drive's per-file rate limit ("had many accesses" -- easy to hit while
        # iterating on the same shared link) get skipped instead of aborting the whole batch.
        remote_files = gdown.download_folder(FOLDER_URL, skip_download=True)
        scan_files = [item for item in remote_files if item.path.lower().endswith((".tif", ".tiff"))]

        downloaded = []
        for item in scan_files:
            local_path = os.path.join(shared_download_dir, os.path.basename(item.path))
            try:
                gdown.download(id=item.id, output=local_path, quiet=False)
                downloaded.append(os.path.basename(local_path))
            except Exception as error:
                print(f"!! Skipping {item.path}: {error}")
        _downloaded_folder_url = FOLDER_URL

    scan_picker = widgets.Dropdown(
        description="Scan:",
        options=sorted(downloaded) or ["(no .tif/.tiff files downloaded)"],
    )
    display(scan_picker)

else:  # "Upload files"
    # Upload scans directly instead of granting any Drive access -- the normal path
    # for tuning against 1-2 images.
    from google.colab import files
    print("Upload the .tif scan(s) to tune/run against:")
    uploaded = files.upload()
    INPUT_DIR = "."
    REFERENCE_SCAN = next(iter(uploaded))
    BATCH_SCANS = list(uploaded)


In [ ]:
#@title Confirm dataset selection
if IN_COLAB and DATA_SOURCE == "Shared folder link":
    # Lock in what got picked in the dropdown above.
    INPUT_DIR = shared_download_dir
    REFERENCE_SCAN = scan_picker.value
    BATCH_SCANS = []  # [] runs every .tif/.tiff in INPUT_DIR; or list filenames for a subset

print(f"INPUT_DIR = {INPUT_DIR!r}")
print(f"REFERENCE_SCAN = {REFERENCE_SCAN!r}")


In [ ]:
#@title Draw ROI boxes
from jupyter_bbox_widget import BBoxWidget
from IPython.display import display

if globals().get("_bbox_reference_scan") == REFERENCE_SCAN and "bbox_widget" in globals():
    # Same reference scan as last time -- reuse the widget instead of rebuilding it and
    # wiping out whatever boxes are already drawn just because this cell re-ran.
    print("Reusing existing ROI drawing for this reference scan.")
    display(bbox_widget)
else:
    # Load the reference scan, downscale to a displayable 8-bit preview for the widget.
    first_scan = tifi.imread(os.path.join(INPUT_DIR, REFERENCE_SCAN))
    first_rgb = cv2.cvtColor(first_scan, cv2.COLOR_GRAY2RGB) if first_scan.ndim == 2 else first_scan[..., :3]
    if first_rgb.dtype != np.uint8:
        first_rgb = cv2.normalize(first_rgb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    preview_scale = 1200 / max(first_rgb.shape[0], first_rgb.shape[1])
    preview = cv2.resize(first_rgb, None, fx=preview_scale, fy=preview_scale, interpolation=cv2.INTER_AREA)
    preview_path = "roi_preview.png"
    cv2.imwrite(preview_path, cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))

    # hide_buttons=True drops the Submit/Skip buttons (we read bbox_widget.bboxes directly,
    # no submit callback needed).
    bbox_widget = BBoxWidget(image=preview_path, classes=["plate"], hide_buttons=True)
    _bbox_reference_scan = REFERENCE_SCAN
    display(bbox_widget)


In [ ]:
#@title Convert boxes to ROI hints
# Convert preview-pixel boxes to image fractions, top-to-bottom, labeled A, B, ...
if not PLATE_ROWS or not PLATE_COLS:
    raise ValueError(
        "PLATE_ROWS/PLATE_COLS are unset -- fill them in on the Config cell "
        "(Section 1) and re-run it before running this cell."
    )

drawn_boxes = sorted(bbox_widget.bboxes, key=lambda box: box["y"])
roi_hints = []
for plate_index, box in enumerate(drawn_boxes):
    roi_hints.append({
        "x": box["x"] / preview.shape[1],
        "y": box["y"] / preview.shape[0],
        "w": box["width"] / preview.shape[1],
        "h": box["height"] / preview.shape[0],
        "rows": PLATE_ROWS,
        "cols": PLATE_COLS,
        "letter": chr(ord("A") + plate_index),
    })
if not roi_hints:
    raise ValueError(
        "No boxes drawn -- draw one box per plate on the bbox widget above, "
        "then re-run this cell."
    )

print(f"Captured {len(roi_hints)} ROI hints:")
for roi in roi_hints:
    print(roi)

In [ ]:
#@title Overlay renderer
def render_overlay(axis, image_rgb, roi_hints, wells, title, plate_boxes=None):
    """Overlay on `axis`: padded search area (yellow), detected plate rect (lime,
    if plate_boxes given), well circles (red) + labels."""
    axis.imshow(image_rgb)
    axis.set_title(title)
    image_height, image_width = image_rgb.shape[:2]
    for roi in roi_hints:
        prior_box = roi_frac_to_px(roi, image_width, image_height)
        search_x0, search_y0, search_x1, search_y1 = pad_box(prior_box, image_width, image_height, MARGIN_FRAC)
        axis.add_patch(plt.Rectangle((search_x0, search_y0), search_x1 - search_x0,
                                     search_y1 - search_y0, fill=False, edgecolor="yellow", linewidth=1.5))
    if plate_boxes:
        for plate_x, plate_y, plate_width, plate_height in plate_boxes:
            axis.add_patch(plt.Rectangle((plate_x, plate_y), plate_width, plate_height,
                                         fill=False, edgecolor="lime", linewidth=2))
    for center_x, center_y, radius, label in wells:
        axis.add_patch(plt.Circle((center_x, center_y), radius, fill=False, edgecolor="red", linewidth=2))
        axis.text(center_x + 10, center_y + 10, label, color="yellow", fontsize=12, fontweight="bold")
    axis.set_aspect("equal")

## 4. Segment colonies per well (Cellpose)

Defines `count_colonies(img_rgb, valid_wells, plate_name, show_plots=True)`: for each detected well, crops to a circle, builds a color-agnostic "distance from background" signal in LAB space, then runs Cellpose to segment/count colonies. Returns a list of `{"Plate", "Well", "Colonies"}` rows. Works off whatever wells the ROI-hinted detection found.

In [ ]:
#@title Colony segmentation
def count_colonies(img_rgb, valid_wells, plate_name, show_plots=True, save_dir=None):
    """Segment colonies in each detected well and return per-well colony counts.

    show_plots displays each well's raw/segmentation panel inline; save_dir (if given)
    writes one PNG per well to disk instead/as well.
    """
    report_data = []
    print(f"Extracting wells and generating colony masks for {plate_name}...\n")

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    dark_margin, bright_margin = L_OUTLIER_MARGIN

    well_progress = tqdm(valid_wells, desc=plate_name, unit="well")
    for (x, y, r, label) in well_progress:
        well_progress.set_postfix(well=label)
        # Crop from img_rgb (known-good color), NOT the raw img
        crop_r = int(r * 0.94)
        y_min, y_max = max(0, y - crop_r), min(img_rgb.shape[0], y + crop_r)
        x_min, x_max = max(0, x - crop_r), min(img_rgb.shape[1], x + crop_r)

        well_crop = img_rgb[y_min:y_max, x_min:x_max].copy()   # RGB

        # Circular mask
        mask = np.zeros(well_crop.shape[:2], dtype="uint8")
        cv2.circle(mask, (well_crop.shape[1] // 2, well_crop.shape[0] // 2), crop_r, 255, -1)
        final_well = cv2.bitwise_and(well_crop, well_crop, mask=mask)   # RGB

        # --- Color-agnostic signal: distance from background in LAB ---
        lab = cv2.cvtColor(final_well, cv2.COLOR_RGB2LAB).astype(np.float32)

        ring = cv2.subtract(mask, cv2.erode(mask, np.ones((60, 60), np.uint8)))
        bg_a = np.median(lab[:, :, 1][ring > 0])
        bg_b = np.median(lab[:, :, 2][ring > 0])
        bg_L = np.median(lab[:, :, 0][ring > 0])

        dist = np.sqrt((lab[:, :, 1] - bg_a) ** 2 + (lab[:, :, 2] - bg_b) ** 2)
        signal = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        signal = cv2.bitwise_and(signal, signal, mask=mask)

        # Non-colony brightness outlier rejection (debris too dark, glint too bright) —
        # see L_OUTLIER_MARGIN in Config for the reasoning. Zeroed out of the Cellpose
        # signal image before segmentation so neither gets counted as a colony.
        dark_mask = (lab[:, :, 0] < (bg_L - dark_margin)) & (mask > 0)
        bright_mask = (lab[:, :, 0] > (bg_L + bright_margin)) & (mask > 0)
        signal[dark_mask | bright_mask] = 0

        # Visual check for L_OUTLIER_MARGIN tuning: red = debris (too dark), cyan = glint (too bright).
        outlier_overlay = final_well.copy()
        outlier_overlay[dark_mask] = [255, 0, 0]
        outlier_overlay[bright_mask] = [0, 255, 255]

        # --- Cellpose ---
        masks_cp, flows, styles = cp_model.eval(
            signal,
            diameter=CELLPOSE_DIAMETER,
            flow_threshold=CELLPOSE_FLOW_THRESHOLD,
            cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
            min_size=CELLPOSE_MIN_SIZE,
            normalize={"normalize": True, "percentile": CELLPOSE_NORMALIZE_PERCENTILE},
        )

        colony_count = int(masks_cp.max())
        report_data.append({
            "Plate": plate_name,
            "Well": label,
            "Colonies": colony_count
        })

        if show_plots or save_dir:
            fig, axes = plt.subplots(1, 3)
            axes[0].imshow(final_well)          # already RGB — no conversion
            axes[0].set_title(f"Raw Well: {label}")
            axes[0].axis('off')

            axes[1].imshow(outlier_overlay)
            axes[1].set_title("Outliers (red=debris, cyan=glint)")
            axes[1].axis('off')

            axes[2].imshow(signal, cmap='magma')
            axes[2].imshow(masks_cp, cmap='nipy_spectral', alpha=0.4)
            axes[2].set_title(f"AI Count: {colony_count} Colonies")
            axes[2].axis('off')

            # number each colony at its centroid so a vibe-check is fast — did Cellpose
            # merge two touching colonies into one, or split one into two?
            colony_ids = np.unique(masks_cp)
            colony_ids = colony_ids[colony_ids != 0]
            if len(colony_ids) > 0:
                centroids = ndimage.center_of_mass(masks_cp, masks_cp, colony_ids)
                for colony_id, (centroid_y, centroid_x) in zip(colony_ids, centroids):
                    axes[2].text(centroid_x, centroid_y, str(int(colony_id)),
                                color='white', fontsize=6, ha='center', va='center',
                                bbox=dict(boxstyle='round,pad=0.1', facecolor='black', alpha=0.5, linewidth=0))

            if save_dir:
                well_path = os.path.join(save_dir, f"{label}.png")
                fig.savefig(well_path, dpi=150, bbox_inches='tight')
            if show_plots:
                plt.show()
            else:
                plt.close(fig)

    return report_data

## 5. Run

Requires `roi_hints` from Section 3c. Set `RUN_MODE` in the Config cell (§1, top) to
**Tune (single image)** to run `REFERENCE_SCAN` alone with plots on, or **Batch (all scans)**
to sweep `INPUT_DIR` (every `.tif`, or just `BATCH_SCANS` if you listed any), writing well/grid
PNGs to `BATCH_OUTPUT_DIR/<plate_name>/`. A batch must be one plate layout, since `roi_hints`
describes a layout. `WELLS_ONLY=True` (Config, §1) stops after well detection (skips Cellpose).

If you change `RUN_MODE`, re-run the Config cell first, then this cell.


In [ ]:
#@title Run
def process_plate(path, show_plots, save_dir=None, wells_only=False):
    plate_name = os.path.basename(path)
    print(f"Reading {plate_name}...")
    img = tifi.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else img[..., :3]

    # ROI-hinted detection (Section 3b/3c).
    valid_wells, plate_boxes = detect_wells_from_rois(img_rgb, roi_hints,
                                                      margin_frac=MARGIN_FRAC, return_boxes=True)

    # Grid overlay (see render_overlay), saved as grid.png per plate.
    if show_plots or save_dir:
        figure, axis = plt.subplots(figsize=(10, 10))
        render_overlay(axis, img_rgb, roi_hints, valid_wells, plate_name, plate_boxes=plate_boxes)
        if save_dir:
            grid_path = os.path.join(save_dir, plate_name, "grid.png")
            os.makedirs(os.path.dirname(grid_path), exist_ok=True)
            figure.savefig(grid_path, dpi=150, bbox_inches="tight")
        if show_plots:
            plt.show()
        else:
            plt.close(figure)

    if wells_only:
        return [{"Plate": plate_name, "Well": label, "x": x, "y": y, "r": r}
                for (x, y, r, label) in valid_wells]

    wells_dir = os.path.join(save_dir, plate_name) if save_dir else None
    return count_colonies(img_rgb, valid_wells, plate_name, show_plots=show_plots, save_dir=wells_dir)


if BATCH_MODE:
    # Wipe any previous batch_output before this run, so a well/grid PNG from a stale run
    # can never get mistaken for output from the current one.
    shutil.rmtree(BATCH_OUTPUT_DIR, ignore_errors=True)
    os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

    # roi_hints describes one plate layout, so a batch is one folder of like-laid-out scans,
    # not a mixed pile. BATCH_SCANS narrows it further when you only want a few files.
    scan_names = BATCH_SCANS or sorted(
        name for name in os.listdir(INPUT_DIR) if name.lower().endswith((".tif", ".tiff")))
    tif_paths = [os.path.join(INPUT_DIR, scan_name) for scan_name in scan_names]
    n_imgs = len(tif_paths)
    print(f"Batch mode: {n_imgs} plates")
    print(f"Writing well/grid images to {BATCH_OUTPUT_DIR}/<plate_name>/\n")
    report_data = []
    for img_idx, path in enumerate(tif_paths, start=1):
        print(f"\n=== [{img_idx}/{n_imgs}] {os.path.basename(path)} ===")
        # WELLS_ONLY is a diagnostic step — show inline even in batch
        report_data.extend(process_plate(
            path, show_plots=WELLS_ONLY, save_dir=BATCH_OUTPUT_DIR, wells_only=WELLS_ONLY
        ))
else:
    report_data = process_plate(os.path.join(INPUT_DIR, REFERENCE_SCAN),
                                show_plots=True, wells_only=WELLS_ONLY)

In [ ]:
#@title Save results
df = pd.DataFrame(report_data)

display(df)

# Save to disk: one combined CSV for batch runs, per-plate CSV for single-image runs;
# "wells" vs "Results" in the name so a wells-only run doesn't overwrite a full run's CSV.
# Batch runs write the CSV inside BATCH_OUTPUT_DIR so it travels with the well/grid PNGs
# in the zip below, instead of sitting separately in the repo root.
suffix = "wells" if WELLS_ONLY else "Results"
if BATCH_MODE:
    csv_name = os.path.join(BATCH_OUTPUT_DIR, f"batch_{suffix}.csv")
else:
    csv_name = f"{os.path.splitext(os.path.basename(REFERENCE_SCAN))[0]}_{suffix}.csv"
df.to_csv(csv_name, index=False)

print(f"💾 Data successfully saved to {csv_name}")

# Zip the whole batch_output folder (well/grid PNGs + CSV) with a timestamp, for sending
# to others — one self-contained file per run instead of a loose folder.
if BATCH_MODE:
    zip_stem = f"{BATCH_OUTPUT_DIR}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    zip_path = shutil.make_archive(zip_stem, "zip", BATCH_OUTPUT_DIR)
    print(f"📦 Batch output zipped to {zip_path}")

# The VM this notebook runs on is ephemeral -- nothing here survives a disconnected/
# recycled runtime unless it's actually downloaded. Trigger a browser download of the
# zip (batch) or CSV (tune) immediately instead of leaving that as a step to forget.
if IN_COLAB:
    from google.colab import files
    files.download(zip_path if BATCH_MODE else csv_name)